In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("umerhaddii/amazon-product-images-dataset-2025")
print("Path to dataset files:", path)

# List contents
import os
for root, dirs, files in os.walk(path):
    print(f"Directory: {root}, Files: {len(files)}")

100%|██████████| 2.65G/2.65G [00:31<00:00, 89.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/umerhaddii/amazon-product-images-dataset-2025/versions/1
Directory: /root/.cache/kagglehub/datasets/umerhaddii/amazon-product-images-dataset-2025/versions/1, Files: 0
Directory: /root/.cache/kagglehub/datasets/umerhaddii/amazon-product-images-dataset-2025/versions/1/images, Files: 15733


In [35]:
# libray that are required for project
import kagglehub
import numpy as np
import pandas as pd
import os
from PIL import Image
import pickle
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.neighbors import NearestNeighbors
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [24]:
# 1. Download dataset
path = kagglehub.dataset_download("umerhaddii/amazon-product-images-dataset-2025")

Using Colab cache for faster access to the 'amazon-product-images-dataset-2025' dataset.


In [36]:
# 2. FIND & CLEAN IMAGES (Robust)
def find_images_robust(dataset_path):
    """Find all valid images with minimal validation"""
    image_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    images = []

    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(image_extensions):
                img_path = os.path.join(root, file)
                try:
                    # Super basic checks only
                    if os.path.getsize(img_path) > 512:  # >512 bytes
                        images.append({
                            'image_path': img_path,
                            'filename': file,
                            'folder': os.path.relpath(root, dataset_path),
                            'size_bytes': os.path.getsize(img_path)
                        })
                except:
                    continue

    df = pd.DataFrame(images)
    print(f" Found {len(df)} image files")
    return df

# Get clean image list
df = find_images_robust(path)

# Generate metadata
df['asin'] = df['filename'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', regex=True)
df['category'] = df['folder'].str.split(os.sep).str[-1]

print(f" Ready: {len(df)} images across {df['category'].nunique()} categories")


 Found 15731 image files
 Ready: 15731 images across 1 categories


In [37]:
# 3. LOAD CNN MODEL
print("\n Loading ResNet50...")
model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

def extract_features_safe(img_path):
    """Bulletproof feature extraction"""
    try:
        # Direct PIL approach (most reliable)
        with Image.open(img_path) as img:
            # Force RGB + resize
            img = img.convert('RGB').resize((224, 224), Image.Resampling.LANCZOS)
            img_array = np.array(img, dtype=np.float32)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = preprocess_input(img_array)
            features = model.predict(img_array, verbose=0)
            return features.flatten()
    except:
        return None


🔥 Loading ResNet50...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [39]:
# 4. BUILD RECOMMENDATION ENGINE (4000 images)
print("\n Building recommendation engine...")
MAX_IMAGES = 4000
catalog_df = df.head(MAX_IMAGES).copy()
catalog_features = []
valid_indices = []
total_valid = 0
print("Extracting features (this takes ~10-15 mins)...")
for i, row in catalog_df.iterrows():
    features = extract_features_safe(row['image_path'])
    if features is not None:
        catalog_features.append(features)
        valid_indices.append(i)
        total_valid += 1

    if (i + 1) % 200 == 0:
        print(f"Processed {i+1}/{MAX_IMAGES} | Valid: {total_valid}")

# Convert to numpy array
catalog_features = np.array(catalog_features)
# Normalize features for cosine similarity with Euclidean metric
catalog_features = catalog_features / np.linalg.norm(catalog_features, axis=1, keepdims=True)
print(f"\n {catalog_features.shape[0]} valid images → {catalog_features.shape} features")
# Build KNN index
print("Building KNN index...")
nn = NearestNeighbors(n_neighbors=6, algorithm='ball_tree', metric='euclidean') # Changed metric to 'euclidean'
nn.fit(catalog_features)


 Building recommendation engine...
Extracting features (this takes ~10-15 mins)...
Processed 200/4000 | Valid: 200
Processed 400/4000 | Valid: 398
Processed 600/4000 | Valid: 597
Processed 800/4000 | Valid: 797
Processed 1000/4000 | Valid: 994
Processed 1200/4000 | Valid: 1193
Processed 1400/4000 | Valid: 1392
Processed 1600/4000 | Valid: 1590
Processed 1800/4000 | Valid: 1789
Processed 2000/4000 | Valid: 1988
Processed 2200/4000 | Valid: 2188
Processed 2400/4000 | Valid: 2387
Processed 2600/4000 | Valid: 2587
Processed 2800/4000 | Valid: 2787
Processed 3000/4000 | Valid: 2986
Processed 3200/4000 | Valid: 3186
Processed 3400/4000 | Valid: 3385
Processed 3600/4000 | Valid: 3585
Processed 3800/4000 | Valid: 3784
Processed 4000/4000 | Valid: 3983

 3983 valid images → (3983, 2048) features
Building KNN index...


NearestNeighbors(algorithm='ball_tree', metric='euclidean', n_neighbors=6)

In [41]:
#5.SAVE PRODUCTION FILES
output_dir = Path("/content/") / "recommendation_engine" # Changed path to a writable location
output_dir.mkdir(exist_ok=True)
# Save features
np.save(output_dir / 'catalog_features.npy', catalog_features)
with open(output_dir / 'knn_index.pkl', 'wb') as f:
    pickle.dump(nn, f)
# Save catalog info
catalog_info = {
    'ids': catalog_df.iloc[valid_indices]['asin'].tolist(),
    'categories': catalog_df.iloc[valid_indices]['category'].tolist(),
    'image_paths': catalog_df.iloc[valid_indices]['image_path'].tolist()
}
with open(output_dir / 'catalog_info.pkl', 'wb') as f:
    pickle.dump(catalog_info, f)

print(f"\nSaved to: {output_dir}")


Saved to: /content/recommendation_engine


In [42]:
# 6. TEST RECOMMENDATIONS
print("\n Testing recommendations...")
# Load saved files
catalog_features = np.load(output_dir / 'catalog_features.npy')
nn_loaded = pickle.load(open(output_dir / 'knn_index.pkl', 'rb'))
catalog_info = pickle.load(open(output_dir / 'catalog_info.pkl', 'rb'))
# Test first image
query_features = catalog_features[0:1]
distances, indices = nn_loaded.kneighbors(query_features)
print("\n Sample recommendations:")
print("Query product:", catalog_info['ids'][0])
print("\nTop 5 similar products:")
for i, idx in enumerate(indices[0][:5]):
    print(f"{i+1}. {catalog_info['ids'][idx]} ({catalog_info['categories'][idx]})")


 Testing recommendations...

 Sample recommendations:
Query product: 516m7Eu6LNL

Top 5 similar products:
1. 516m7Eu6LNL (images)
2. 61CTW6tyotL (images)
3. 51l4Gl63zIL (images)
4. 71twHMXse8L (images)
5. 61AZmK-uOfL (images)
